In [1]:
import re
from glob import glob

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from jax import numpy as jnp
from scipy import linalg
from scipy import special as ss

## Plot functions

In [19]:
custom_params = {
  "font.family": "serif",
  "font.weight": "light",
  "font.serif": "cmr10",
  "mathtext.fontset": "stix",
  "axes.labelsize": 14,
  "text.usetex": False,
}
sns.set_theme(style="ticks", context="paper", rc=custom_params)


def bin_plot(results, mag_bins, dist_bins, ax, cmap, scale):
  plot = sns.heatmap(
    results,
    annot=True,
    fmt=".1f",
    ax=ax,
    cmap=cmap,
    linewidth=1.0,
    mask=np.isnan(results),
    # annot_kws={'color':'black'},
    vmin=scale[0],
    vmax=scale[1],
  )
  plot.set_xticks(np.arange(len(mag_bins)))
  plot.set_xticklabels(mag_bins)
  plot.set_yticks(np.arange(len(dist_bins)))
  plot.set_yticklabels(dist_bins)
  plot.invert_yaxis()
  plot.set_xlabel("Magnitude bin")
  plot.set_ylabel("Distance bin [km]")
  fig = plot.get_figure()
  fig.tight_layout()
  return fig

## Functions

In [3]:
def get_magnitudes_and_distances(waveforms):
  meta = waveforms["meta"][:]
  magnitudes = meta[:, 0]
  distances = jnp.sqrt(
    jnp.square(meta[:, 6] - meta[:, 2])
    + jnp.square(meta[:, 7] - meta[:, 3])
    + jnp.square(meta[:, 5] - meta[:, 4] * 1e-3)
  )
  return magnitudes, distances

In [4]:
def frechet_distance(x: np.ndarray, y: np.ndarray, isotropic=False, eps=1e-6):
  mu_x = np.nanmean(x, axis=0)
  mu_y = np.nanmean(y, axis=0)

  if isotropic:
    std_x = np.nanstd(x, axis=0)
    std_y = np.nanstd(y, axis=0)
    return np.sum((mu_x - mu_y) ** 2) + np.sum((std_x - std_y) ** 2)

  cov_x = np.cov(x, rowvar=False)
  cov_y = np.cov(y, rowvar=False)

  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
  if not np.isfinite(covmean).all():
    print(
      f"fid calculation produces singular product; adding {eps} to diagonal of cov estimates"
    )
    offset = np.eye(cov_x.shape[0]) * eps
    covmean = linalg.sqrtm((cov_x + offset) @ (cov_y + offset))

  # Numerical error might give slight imaginary component
  if np.iscomplexobj(covmean):
    if not np.allclose(np.diagonal(covmean).imag, 0, atol=1e-3):
      raise ValueError(f"Imaginary component {np.max(np.abs(covmean.imag))}")
    covmean = covmean.real

  return (
    np.sum((mu_x - mu_y) ** 2)
    + np.trace(cov_x)
    + np.trace(cov_y)
    - 2 * np.trace(covmean)
  )


def compute_fid(target_hat_embedding, target_embedding):
  fid = frechet_distance(
    target_hat_embedding, target_embedding, isotropic=False
  )
  return fid


def compute_fid_per_label(waveforms, mag_bins, dist_bins):
  mag, dist = get_magnitudes_and_distances(waveforms)
  results = []
  for i in range(len(dist_bins) - 1):
    results.append([])
    for j in range(len(mag_bins) - 1):
      mask = (dist >= dist_bins[i]) & (dist < dist_bins[i + 1])
      mask &= (mag >= mag_bins[j]) & (mag < mag_bins[j + 1])
      try:
        fid = compute_fid(
          waveforms["target_hat_classifier_embedding"][mask],
          waveforms["target_classifier_embedding"][mask],
        )
      except Exception:
        fid = np.nan
      results[i].append(fid)
  return results

In [5]:
def compute_is(target_hat_preds, eps=1e-50):
  probs = ss.softmax(target_hat_preds.astype(np.float64), axis=-1)

  probs = np.maximum(probs, eps)
  probs /= probs.sum(axis=1, keepdims=True)
  p_y = probs.mean(axis=0)

  log_is = np.mean(np.sum(probs * (np.log(probs) - np.log(p_y)), axis=1))
  return np.exp(log_is)

In [6]:
def get_label(waveforms, distance_bins, magnitude_bins):
  magnitudes, distances = get_magnitudes_and_distances(waveforms)
  labels = (
    (jnp.digitize(distances, distance_bins) - 1) * (len(magnitude_bins) - 1)
    + jnp.digitize(magnitudes, magnitude_bins)
    - 1
  )
  return labels


def accuracy(logits, labels):
  return (logits.argmax(axis=-1) == labels).mean() * 100


def compute_accuracy(waveforms, distance_bins, magnitude_bins):
  labels = get_label(waveforms, distance_bins, magnitude_bins)
  acc_data = accuracy(waveforms["target_preds"][:], labels)
  acc_syn = accuracy(waveforms["target_hat_preds"][:], labels)
  return {"acc_syn": acc_syn, "acc_data": acc_data}

In [7]:
def spectral_density(signal):
  sd = np.abs(np.fft.rfft(signal, axis=-1))
  log_sd = np.log(np.clip(sd, 1e-8, None))
  return log_sd


def compute_asd_fd(target, target_hat, cut=8033):
  asd_preds = spectral_density(target[:, :cut])
  asd_targets = spectral_density(target_hat[:, :cut])
  fd = frechet_distance(asd_preds, asd_targets, isotropic=True)
  return fd


def compute_asd_fd_per_label(
  waveforms, distance_bins, magnitude_bins, cut=8033
):
  mag, dist = get_magnitudes_and_distances(waveforms)
  all_results = {}
  for c in range(3):
    results = []
    for i in range(len(distance_bins) - 1):
      results.append([])
      for j in range(len(magnitude_bins) - 1):
        mask = (dist >= distance_bins[i]) & (dist < distance_bins[i + 1])
        mask &= (mag >= magnitude_bins[j]) & (mag < magnitude_bins[j + 1])
        try:
          fid = compute_asd_fd(
            waveforms["target_hat"][mask, :cut, c],
            waveforms["target"][mask, :cut, c],
          )
        except Exception:
          fid = np.nan
        results[i].append(fid)
    all_results[c] = results
  return all_results

In [8]:
def geometric_mean(waveform, cut=8033):
  obs_max = np.max(np.abs(waveform["target"][:, :cut, :2]), axis=1)
  syn_max = np.max(np.abs(waveform["target_hat"][:, :cut, :2]), axis=1)

  obs_geom = np.mean(np.log10(obs_max), axis=1)
  syn_geom = np.mean(np.log10(syn_max), axis=1)

  ratio = obs_geom - syn_geom
  return {
    "mean": np.abs(np.mean(ratio)),
    "median": np.abs(np.median(ratio)),
    "std": np.std(ratio),
  }

## Config

In [9]:
distance_bins = [0.0, 25.0, 50.0, 75.0, 125.0]
magnitude_bins = [1.0, 2.0, 2.5, 3.5, 4.5, 6.6]

## Visualization

In [10]:
model_nice = {
  "ddm-dit-fm": "FM (DiT)",
  "ddm-unet-fm": "FM (UNet)",
  "ddm-dit-edm": "EDM (DiT)",
  "ddm-unet-edm": "EDM (UNet)",
  "ddm-dit_patch_8-fm": "FM (DiT 8x8)",
  "ddm-dit_patch_4-fm": "FM (DiT 4x4)",
  "ddm-dit_small-fm": "FM (small DiT)",
  "ddm-dit_patch_4-edm": "EDM (DiT 4x4)",
  "ddm-dit_small-edm": "EDM (small DiT)",
  "wgan-pix2pix": "WGAN-GP (pix2pix)",
  "cd-dit-edm": "CD (DiT)",
}
model_nfe = {
  "wgan-pix2pix": 1,
  "ddm-unet-edm": 50,
  "ddm-dit-edm": 50,
  "ddm-dit-fm": 25,
  "ddm-dit_patch_4-fm": 25,
  "ddm-unet-fm": 25,
  "cd-dit-edm": 2,
  "ddm-dit_small-fm": 25,
  "ddm-dit_patch_4-edm": 50,
  "ddm-dit_small-edm": 50,
}

### Bin plots FID

In [21]:
def plot_fid(file, plot_file, fid, magnitude_bins, distance_bins):
  cmap = sns.color_palette("Reds", as_cmap=False, desat=1, n_colors=30).as_hex()
  white_red_cmap = sns.blend_palette([cmap[0], cmap[-1]], as_cmap=True)

  model_name = reg.match(file).group(1)
  fig, ax = plt.subplots(figsize=(4, 3))
  (
    _,
    bin_plot(
      fid, magnitude_bins, distance_bins, ax, white_red_cmap, scale=[0, 1300]
    ),
  )
  plt.savefig(plot_file)
  plt.close()

In [22]:
reg = re.compile("^.*evaluation-(.*)-small_ae.h5$")
fid_files = glob("../../../results/workdir/outputs/eval*h5")

for file in sorted(fid_files):
  model_name = reg.match(file).group(1)
  plot_file = f"../../../results/workdir/figures/fid-{model_name}.pdf"
  # if os.path.isfile(plot_file):
  #    continue
  print(f"doing {file}")
  with h5py.File(file, "r") as test_waveforms:
    fid = compute_fid_per_label(test_waveforms, magnitude_bins, distance_bins)
    plot_fid(file, plot_file, fid, magnitude_bins, distance_bins)

doing ../../../results/workdir/outputs/evaluation-cd-dit-edm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/2065612112.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, ax = plt.subplots(figsize=(4, 3))
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)


doing ../../../results/workdir/outputs/evaluation-ddm-dit-edm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/2065612112.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, ax = plt.subplots(figsize=(4, 3))
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)


doing ../../../results/workdir/outputs/evaluation-ddm-dit-fm-small_ae.h5


...

/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/2065612112.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, ax = plt.subplots(figsize=(4, 3))
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)


doing ../../../results/workdir/outputs/evaluation-ddm-unet-fm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/2065612112.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, ax = plt.subplots(figsize=(4, 3))
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)


doing ../../../results/workdir/outputs/evaluation-wgan-pix2pix-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: DeprecationWarning: The `disp` argument is deprecated and will be removed in SciPy 1.18.0.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/349393905.py:13: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean, _ = linalg.sqrtm(cov_x @ cov_y, disp=False)
/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/2065612112.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, ax = plt.subplots(figsize=(4, 3))
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)


### Bin plots ASD-FD

In [23]:
def plot_asd_fd(file, plot_file, asd, magnitude_bins, distance_bins):
  cmap = sns.color_palette(
    "Blues", as_cmap=False, desat=1, n_colors=30
  ).as_hex()
  blue_white_cmap = sns.blend_palette([cmap[0], cmap[-1]], as_cmap=True)

  model_name = reg.match(file).group(1)
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
  for c, tit in zip(range(3), ["East - West", "North - South", "Vertical"]):
    (
      _,
      bin_plot(
        asd[c],
        magnitude_bins,
        distance_bins,
        axes[c],
        blue_white_cmap,
        scale=[None, None],
      ),
    )
    axes[c].set_title(f"{tit}", fontsize=16)
  plt.savefig(plot_file)
  plt.close()

In [24]:
reg = re.compile("^.*evaluation-(.*)-small_ae.h5$")
asd_files = glob("../../../results/workdir/outputs/eval*h5")

for file in sorted(asd_files):
  model_name = reg.match(file).group(1)
  plot_file = f"../../../results/workdir/figures/asd-{model_name}.pdf"
  # if os.path.isfile(plot_file):
  #     continue
  print(f"doing {file}")
  with h5py.File(file, "r") as test_waveforms:
    asd = compute_asd_fd_per_label(
      test_waveforms, distance_bins=distance_bins, magnitude_bins=magnitude_bins
    )
    plot_asd_fd(file, plot_file, asd, magnitude_bins, distance_bins)

doing ../../../results/workdir/outputs/evaluation-cd-dit-edm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/3515704348.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mes

doing ../../../results/workdir/outputs/evaluation-ddm-dit-edm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/3515704348.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mes

doing ../../../results/workdir/outputs/evaluation-ddm-dit-fm-small_ae.h5


...

/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/3515704348.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mes

doing ../../../results/workdir/outputs/evaluation-ddm-unet-fm-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/3515704348.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mes

doing ../../../results/workdir/outputs/evaluation-wgan-pix2pix-small_ae.h5


/var/folders/w8/7mc8k9m916qgh982xqxfgsr00000gn/T/ipykernel_52700/3515704348.py:6: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  fig, axes = plt.subplots(figsize=(12, 3), ncols=3)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mesh, cax, ax, **self.cbar_kws)
/Users/simon/PROJECTS/2024-highfem/src/01-gm1/.venv/lib/python3.11/site-packages/seaborn/matrix.py:316: UserWarning: cmr10 font should ideally be used with mathtext, set axes.formatter.use_mathtext to True
  cb = ax.figure.colorbar(mes

### Tables

In [23]:
fids = {}
gms = {}
ids = {}
accs = {}
asd_fds = {}
for file in files:
  test_waveforms = h5py.File(file, "r")
  fids[file] = compute_fid(test_waveforms)
  ids[file] = compute_is(test_waveforms)
  gms[file] = geometric_mean(test_waveforms)
  accs[file] = compute_accuracy(test_waveforms)
  asd_fds[file] = compute_asd_fd(test_waveforms)

In [24]:
reg = re.compile("^.*evaluation-(.*)-small_ae.h5$")
for file in files:
  mtch = reg.match(file).group(1)
  models[file] = model_nice[mtch]
  nfe[file] = model_nfe[mtch]

In [ ]:
res_df = pd.DataFrame(
  {
    "Model": [models[file] for file in files],
    "NFE": [nfe[file] for file in files],
    "FID": [fids[file] for file in files],
    "IS": [ids[file] for file in files],
    "GM": [gms[file]["mean"] for file in files],
    "Accuracy": [accs[file]["acc_syn"] for file in files],
    "ASD": [np.mean(list(asd_fds[file].values())) for file in files],
  }
)

res_df = res_df.loc[:, ["Model", "NFE", "FID", "IS", "Accuracy", "ASD", "GM"]]
res_df.ASD = np.array(res_df.ASD.values, dtype=np.float32)
res_df.Accuracy = np.array(res_df.Accuracy.values, dtype=np.float32)
res_df.IS = np.array(res_df.IS.values, dtype=np.float32)
res_df = res_df.sort_values("Model")

In [26]:
float_formats = {
  "GM": "{:.3f}",
  "FID": "{:.2f}",
  "Accuracy": "{:.2f}",
  "ASD": "{:.2f}",
  "IS": "{:.2f}",
}


def format_col(val, col_name):
  fmt = float_formats.get(col_name, "{:.2f}")
  return np.array(fmt.format(val), dtype=np.float64) if fmt else val


for col in res_df.columns:
  if col in float_formats:
    res_df[col] = res_df[col].apply(lambda x: format_col(x, col))

In [27]:
higher_better = ["Accuracy", "IS"]
lower_better = ["FID", "ASD", "GM"]


def highlight_best(s, higher=True):
  if higher:
    is_best = s == s.max()
  else:
    is_best = s == s.min()
  return [
    "\\textbf{" + str(v) + "}" if b else str(v) for v, b in zip(s, is_best)
  ]


for col in higher_better:
  res_df[col] = highlight_best(res_df[col], higher=True)
for col in lower_better:
  res_df[col] = highlight_best(res_df[col], higher=False)

In [28]:
xx = res_df.to_latex(
  index=False,
  caption="Model Performance",
  label="tab:model-performance",
  escape=False,
)
print(xx)

\begin{table}
\caption{Model Performance}
\label{tab:model-performance}
\begin{tabular}{lrlllll}
\toprule
Model & NFE & FID & IS & Accuracy & ASD & GM \\
\midrule
CD (DiT) & 2 & \textbf{34.82} & 10.34 & \textbf{78.95} & \textbf{95.69} & \textbf{-0.013} \\
EDM (DiT 4x4) & 50 & 50.86 & 11.07 & 76.94 & 116.93 & 0.079 \\
EDM (DiT) & 50 & 55.5 & 11.27 & 76.82 & 114.52 & 0.08 \\
EDM (UNet) & 50 & 54.62 & 11.18 & 77.21 & 146.63 & 0.082 \\
EDM (small DiT) & 50 & 53.88 & 11.24 & 76.88 & 124.86 & 0.074 \\
FM (DiT 4x4) & 25 & 58.35 & 11.24 & 75.57 & 174.06 & 0.085 \\
FM (DiT) & 25 & 58.12 & \textbf{11.47} & 76.11 & 126.45 & 0.078 \\
FM (UNet) & 25 & 58.26 & 11.23 & 76.56 & 153.52 & 0.075 \\
FM (small DiT) & 25 & 56.04 & 11.21 & 76.5 & 141.38 & 0.078 \\
WGAN-GP (pix2pix) & 1 & 49.26 & 9.74 & 74.02 & 129.49 & 0.02 \\
\bottomrule
\end{tabular}
\end{table}

